In [88]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import softmax 
from scipy.special import logsumexp
from itertools import product

In [89]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [90]:
#DNA - 60 sequences
seqs = ["TTGTTT", "CCTAAT", "TTCCCC", "GGGTTT", "CCTAAG", "AATCCT", "CCCAAA", "TTGCCA", "GGTTTG", "AACAAG", "GGGGGA", "AACTTC", "CCCGGG", "CAACCA", "AAACCT", "TTCTTT", "GGTTAG", "TTGGGC", "GGTTTC", "CCAAAC", "CCAGGC", "GGAGGG", "GTTTTA", "GGGTTA", "TTTGGT", "AAAAAC", "CCTGTC", "AAATAG", "TGGGGC", "GGAAAC", "GGTTTG", "TTCCCC", "CCCTTT", "TTAGCG", "TTAGGG", "GGATTT", "TTGTGT", "CCAAAA", "TTTGGA", "AGAAAC", "GGCGGG", "CAATTT",
 "GGTCCA", "CCGTTT", "TTCAAG", "TTCGGG", "CCAGGT", "CCCGGA","TTACCC", "GGTAAG", "CCGGGA", "TTGAAT", "AAATTA", "GGGTTC", "CGTAAC", "CCCAAG", "TTCCCA", "TTCCGG", "GGTAAT", "CCGTTT"]

a = "ACGT"
#prior
sigma_h_squared = np.inf  #np.inf = no prior on h
sigma_J_squared = 1
q = len(a)

In [91]:
# transfer letter to numbers
def encode(seqs, a):
    encoded_sequences = []
    for sequence in seqs:
        encoded_sequence = []
        for letter in sequence:
            encoded_sequence.append(a.index(letter))
        encoded_sequences.append(encoded_sequence)
    return np.array(encoded_sequences)
encoded_sequences = encode(seqs, a)
N, L = encoded_sequences.shape

In [92]:
# all possible sequences - 4^6
all_sequences = np.array(list(product(range(q), repeat=L)))
pairs = [(i, j) for i in range(L) for j in range(i + 1, L)]

In [ ]:
#E(X)
#option1 - potts


#E(X) = sum_i h_i(x_i) + sum_(i!=j) J_{ij}(x_i, x_j)
#theta = {h_i, J_ij}

n_potts = L * q + L * L * q * q

def potts_energy(seqs, theta):
    h = theta[:L*q].reshape(L,q)
    J = theta[L*q:].reshape(L,L,q,q)
    E = h[np.arange(L), seqs].sum(axis=1)
    for i, j in pairs:
        E += J[i, j, seqs[:, i], seqs[:, j]]
    return E

#gradient for potts_energy E -> likelihood
#dE/dh_i(k) = 1 if x_i = k, 0 otherwise
#dE/dJ_{ij}(k, l) = 1 if x_i =k and x_j = l, 0 otherwise
def potts_energy_gradient(seqs):
    M = len(seqs)
    rows = np.arange(M)
    gradient_h = np.zeros((M, L, q))
    gradient_J = np.zeros((M, L, L, q, q))
    for i in range(L):
        gradient_h[rows, i, seqs[:, i]] = 1
    for i, j in pairs:
        gradient_J[rows, i, j, seqs[:, i], seqs[:, j]] = 1
    return np.concatenate([gradient_h.reshape(M, -1), gradient_J.reshape(M, -1)], axis=1)

#prior
#log P(h) + log P(J) = -sum h^2 / (2 sigma_h^2) - sum J^2 / (2 sigma_J^2)
def potts_prior(theta):
    h, J = theta[:L*q], theta[L*q:]
    return -0.5 * np.sum(h**2)/sigma_h_squared - 0.5 * np.sum(J**2)/sigma_J_squared

#d/dh = -h / sigma_h^2
#d/dJ = -J / sigma_J^2
def potts_prior_gradient(theta):
    h, J = theta[:L*q], theta[L*q:]
    return np.concatenate([-h/sigma_h_squared, -J/sigma_J_squared])

In [94]:
#option 2 - kernel method


#K(x, u) = sigma_E^2 * exp(-H(x, u) / c)
#H: hamming distance between sequences
#sigma_e^2 and c are fixed right now, but will be updated later
c = 1
sigma_E_squared = 1

def K(seqs1, seqs2):
    H = np.sum(seqs1[:, None, :] != seqs2[None, :, :], axis=2)
    return sigma_E_squared * np.exp(-H / c)


#inducing sequences u
U = np.unique(encoded_sequences, axis=0)
K_uu = K(U, U)


#E(X) = sum_n K(X, U) alpha_u
#theta = alpha
n_kernel = len(U)
def kernel_energy(seqs, alpha):
    return K(seqs, U) @ alpha

#gradient for kernel energy E -> likelihood
#dE/dalpha_u = K(X, U)
def kernel_energy_gradient(seqs):
    return K(seqs, U)

#prior
#alpha = N(0, K_uu -1)
#log p(alpha) = -1/2 alpha transpose K_uu alpha
def kernel_prior(alpha, K_uu):
    return -0.5 * alpha.T @ K_uu @ alpha

#gradient for the kernel prior
def kernel_prior_gradient(alpha):
    return -K_uu @ alpha

In [95]:
#likelihood
#option1 - likelihood

#log p = sum_n E(x_n) - N log Z
def log_likelihood(theta, energy, encoded_sequences, all_sequences, N):
    log_Z = logsumexp(energy(all_sequences, theta))
    return np.sum(energy(encoded_sequences, theta)) - N * log_Z

#gradient = sum_n dE(x_n) - N sum_x p(x) dE(x)
def log_likelihood_gradient(theta, energy, energy_gradient, encoded_sequences, all_sequences, N):
    E_all = energy(all_sequences, theta)
    P = softmax(E_all)   #p(x) = exp(E(x)) / Z
    return energy_gradient(encoded_sequences).sum(axis=0) - N * P @ energy_gradient(all_sequences)

In [96]:
#option 2 - pseudolikelihood


#pseudolikelihood for potts model

#conditional probability of x_i given x_-i
#E(X_P|X_-P) = h_i(x_i) + sum_(j != i) J_ij(x_i, x_j)
def potts_conditional_energy(seqs, theta):
    h = theta[:L*q].reshape(L,q)
    J = theta[L*q:].reshape(L,L,q,q)
    E_conditional_probability = np.zeros((len(seqs), L, q))
    #h(i) for every letter a
    for i in range(L):
        E_conditional_probability[:, i, :] = h[i]
    #adding J_ij(a, x_j) for every other position j
        for j in range(L):
            if i!=j:
                E_conditional_probability[:, i, :] += J[i, j, :, seqs[:, j]]
    return E_conditional_probability

#log pseudolikelihood = sum_n sum_i log p(x_i | x_-i)
#                     = sum_n sum_i [E(x_i | x_-i) - log Z_i]
#log Z_i = log sum_a exp(E(a | x_-i))
def potts_log_pseudolikelihood(theta, encoded_sequences):
    E_conditional_probability = potts_conditional_energy(encoded_sequences, theta)
    log_Z = logsumexp(E_conditional_probability, axis=2)
    #E(x_i | x_-i): only the letter that is at position i
    E_true = np.zeros((len(encoded_sequences), L))
    for n in range(len(encoded_sequences)):
        for i in range(L):
            E_true[n, i] = E_conditional_probability[n, i, encoded_sequences[n, i]]
    return np.sum(E_true) - np.sum(log_Z)

#single site variants of one sequence x at position p (like 00, 10 in epistasis)
#only position p is changed, everything else stays the same
#  variants[a] = x with x_p = a,  for every letter a in the alphabet
def single_site_variants(x, p):
    variants = np.array([x] * q)
    for a in range(q):
        variants[a, p] = a
    return variants


#pseudolikelihood for kernel model
#use the energy of the whole variant F_a
#log p(x_p | x_-p) = F_true - log sum_a exp(F_a)
def kernel_log_pseudolikelihood(theta, encoded_sequences):
    total = 0
    for x in encoded_sequences:
        for p in range(L):
            F = kernel_energy(single_site_variants(x, p), theta)   #F_a for every letter a
            total += F[x[p]] - logsumexp(F)
    return total


#gradient for pseudolikelihood (potts & kernel)
#  d/dtheta [E(x_p | x_-p) - log Z_p] = dF_true - sum_a P(a | x_-p) dF_a
#  P(a | x_-p) = exp(F_a) / sum_a exp(F_a)
def log_pseudolikelihood_gradient(theta, energy, energy_gradient, encoded_sequences):
    gradient = np.zeros_like(theta)
    for x in encoded_sequences:
        for p in range(L):
            variants = single_site_variants(x, p)
            F = energy(variants, theta)           
            dF = energy_gradient(variants)        
            P = softmax(F)                        
            #first term: the true letter, F_true is x itself
            gradient += dF[x[p]]
            #second term: sum_a P(a | x_-p) dF_a
            for a in range(q):
                gradient -= P[a] * dF[a]
    return gradient

In [97]:
#ML: maximize log likelihood
#BI: maximize log likelihood + log prior
def fit(energy_type, likelihood_type, inference):
    if energy_type == "potts":
        energy = potts_energy
        energy_gradient = potts_energy_gradient
        n_parameters = n_potts
    else:
        energy = kernel_energy
        energy_gradient = kernel_energy_gradient
        n_parameters = n_kernel

    #minimize -(log likelihood + log prior)
    def f(theta):
        if likelihood_type == "likelihood":
            value = log_likelihood(theta, energy, encoded_sequences, all_sequences, N)
        elif energy_type == "potts":
            value = potts_log_pseudolikelihood(theta, encoded_sequences)
        else:
            value = kernel_log_pseudolikelihood(theta, encoded_sequences)

        if inference == "BI" and energy_type == "potts":
            value += potts_prior(theta)
        if inference == "BI" and energy_type == "kernel":
            value += kernel_prior(theta, K_uu)
        return -value

    def df(theta):
        if likelihood_type == "likelihood":
            g = log_likelihood_gradient(theta, energy, energy_gradient, encoded_sequences, all_sequences, N)
        else:
            g = log_pseudolikelihood_gradient(theta, energy, energy_gradient, encoded_sequences)

        if inference == "BI" and energy_type == "potts":
            g += potts_prior_gradient(theta)
        if inference == "BI" and energy_type == "kernel":
            g += kernel_prior_gradient(theta)
        return -g

    result = minimize(f, np.zeros(n_parameters), jac=df, method='L-BFGS-B')
    return result.x, energy

In [98]:
#strength_ij = sqrt(sum_(k,l) J_ij(k,l)^2), potts only
def coupling_strength(theta):
    J = theta[L*q:].reshape(L, L, q, q)
    strength = np.sqrt(np.sum(J**2, axis=(2, 3)))
    return strength + strength.T

#epistasis between position i and j on a background sequence x
#change x_i to a2 and x_j to b2
#epsilon = (E11 - E01) - (E10 - E00)
#potts: epsilon does not depend on the background
def epistasis(energy, theta, background, i, j, a2, b2):
    seqs = np.array([background, background, background, background])
    seqs[1, i] = a2
    seqs[2, j] = b2
    seqs[3, i] = a2
    seqs[3, j] = b2
    E00, E10, E01, E11 = energy(seqs, theta)
    return (E11 - E01) - (E10 - E00)

#epsilon
#over all data sequences as background
#and all changes at position i and j
def epistasis_matrix(energy, theta):
    strength = np.zeros((L, L))
    for i, j in pairs:
        values = []
        for x in encoded_sequences:
            for a2 in range(q):
                for b2 in range(q):
                    if a2 != x[i] and b2 != x[j]:
                        values.append(abs(epistasis(energy, theta, x, i, j, a2, b2)))
        strength[i, j] = np.mean(values)
        strength[j, i] = strength[i, j]
    return strength

In [100]:
np.set_printoptions(precision=2, suppress=True)

for energy_type in ["potts", "kernel"]:
    for likelihood_type in ["likelihood", "pseudolikelihood"]:
        for inference in ["ML", "BI"]:
            theta, energy = fit(energy_type, likelihood_type, inference)

            #average log p(x_n)
            average_log_p = np.mean(energy(encoded_sequences, theta)) - logsumexp(energy(all_sequences, theta))

            print(energy_type, likelihood_type, inference)
            print("average log p(x) =", round(average_log_p, 3))
            if energy_type == "potts":
                print("coupling strength:")
                print(coupling_strength(theta))
            print("epistasis:")
            print(epistasis_matrix(energy, theta))
            print()

potts likelihood ML
average log p(x) = -4.857
coupling strength:
[[  0.   100.21  42.55  17.75  26.48  30.4 ]
 [100.21   0.    44.16  32.8   27.44  29.91]
 [ 42.55  44.16   0.    41.84  41.58  34.11]
 [ 17.75  32.8   41.84   0.   100.57  24.47]
 [ 26.48  27.44  41.58 100.57   0.    24.61]
 [ 30.4   29.91  34.11  24.47  24.61   0.  ]]
epistasis:
[[ 0.   73.51 22.61  9.41 14.78 15.67]
 [73.51  0.   24.27 17.59 13.34 15.92]
 [22.61 24.27  0.   23.63 23.58 10.85]
 [ 9.41 17.59 23.63  0.   68.25 13.42]
 [14.78 13.34 23.58 68.25  0.   13.36]
 [15.67 15.92 10.85 13.42 13.36  0.  ]]

potts likelihood BI
average log p(x) = -5.498
coupling strength:
[[0.   4.08 1.42 1.14 1.25 1.27]
 [4.08 0.   1.46 1.36 1.18 0.96]
 [1.42 1.46 0.   0.97 1.07 1.66]
 [1.14 1.36 0.97 0.   4.05 1.24]
 [1.25 1.18 1.07 4.05 0.   1.27]
 [1.27 0.96 1.66 1.24 1.27 0.  ]]
epistasis:
[[0.   3.04 0.81 0.67 0.71 0.72]
 [3.04 0.   0.74 0.7  0.58 0.49]
 [0.81 0.74 0.   0.54 0.6  0.83]
 [0.67 0.7  0.54 0.   2.97 0.7 ]
 [0.71 0.5